# Lecture 11: Training Neural Networks II

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 11, with material from Stanford CS231n 2026, Lectures 3 and 6.*

Lecture 10 covered the setup before training. This lecture covers **what happens during and after training**.

**What you'll learn**
1. Learning-rate schedules (a quick recap).
2. **How long to train:** early stopping.
3. **Choosing hyperparameters:** grid vs. random search, and a step-by-step recipe.
4. Reading **learning curves** to diagnose problems.
5. **After training:** model ensembles and **transfer learning**.
6. Training with **large batches** on many GPUs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

**Setup:** the digits, and a small network we'll reuse throughout. `train` records the **learning curves** (training loss after every step; training and validation accuracy after every epoch).

In [ ]:
digits = load_digits()
X_all = digits.data / 16.0 - 0.3
y_all = digits.target
order = np.random.default_rng(0).permutation(len(X_all))
X_train, y_train = X_all[order[:1000]], y_all[order[:1000]]
X_val, y_val = X_all[order[1000:1300]], y_all[order[1000:1300]]
X_test, y_test = X_all[order[1300:]], y_all[order[1300:]]

In [ ]:
class Network:
    # A fully connected ReLU network, e.g. sizes = [64, 128, 10].
    def __init__(self, sizes, reg=1e-4, init_scale=1.0, seed=0):
        rng = np.random.default_rng(seed)
        self.W = [init_scale * rng.normal(0, np.sqrt(2 / a), size=(a, b)) for a, b in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros(b) for b in sizes[1:]]
        self.reg = reg
        self.frozen = []                                  # indices of layers that should NOT be updated

    def scores(self, X):
        h = X
        for i in range(len(self.W) - 1):
            h = np.maximum(0, h @ self.W[i] + self.b[i])
        return h @ self.W[-1] + self.b[-1]

    def accuracy(self, X, y):
        return np.mean(np.argmax(self.scores(X), axis=1) == y)

    def loss(self, X, y):
        s = self.scores(X)
        probs = np.exp(s - s.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
        return -np.mean(np.log(probs[np.arange(len(y)), y] + 1e-12))

In [ ]:
def train_step(net, X, y, lr):
    # One step of gradient descent on a minibatch. Returns the loss.
    inputs, pre_acts, h = [], [], X
    for i in range(len(net.W)):                          # forward, remembering what backward needs
        inputs.append(h)
        z = h @ net.W[i] + net.b[i]
        pre_acts.append(z)
        h = np.maximum(0, z) if i < len(net.W) - 1 else z
    probs = np.exp(h - h.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
    loss = -np.mean(np.log(probs[np.arange(len(y)), y] + 1e-12))
    probs[np.arange(len(y)), y] -= 1
    d = probs / len(y)
    for i in reversed(range(len(net.W))):                # backward, updating as we go
        if i < len(net.W) - 1:
            d = d * (pre_acts[i] > 0)
        grad_W = inputs[i].T @ d + 2 * net.reg * net.W[i]
        grad_b = d.sum(axis=0)
        d = d @ net.W[i].T
        if i not in net.frozen:
            net.W[i] -= lr * grad_W
            net.b[i] -= lr * grad_b
    return loss

def train(net, X, y, epochs, batch_size=32, lr=0.05, lr_schedule=None, seed=0):
    rng = np.random.default_rng(seed)
    history = {"loss": [], "train_acc": [], "val_acc": []}
    step = 0
    for epoch in range(epochs):
        for batch in np.array_split(rng.permutation(len(X)), max(1, len(X) // batch_size)):
            current_lr = lr_schedule(step) if lr_schedule else lr
            with np.errstate(all="ignore"):
                history["loss"].append(train_step(net, X[batch], y[batch], current_lr))
            step += 1
        history["train_acc"].append(net.accuracy(X, y))
        history["val_acc"].append(net.accuracy(X_val, y_val))
    return history

## 1. Learning-rate schedules (recap)

From Lecture 4: **start with a large learning rate, then decay it** (step, cosine, linear, or inverse square root), often with a short **warmup** at the start. Johnson's advice: first find a good **constant** learning rate, *then* add a schedule.

## 2. How long to train? Early stopping

**Train for a long time, but keep the model snapshot that worked best on the validation set.** It's always a good idea.

To see why, we need a reason for longer training to hurt. Real datasets have one: **some labels are wrong**. A network first learns the real patterns, then later starts **memorizing the wrong labels**. Let's flip 30% of the labels in a 200-image training set (the validation labels stay correct):

In [ ]:
rng = np.random.default_rng(0)
X_small = X_train[:200]
y_noisy = y_train[:200].copy()
flip = rng.random(200) < 0.3
y_noisy[flip] = (y_noisy[flip] + rng.integers(1, 10, flip.sum())) % 10      # replace with a different, wrong digit
print(f"{flip.sum()} of 200 labels are wrong")

net = Network([64, 256, 256, 10], reg=0.0)
history = train(net, X_small, y_noisy, epochs=150, batch_size=50, lr=0.1)
best_epoch = int(np.argmax(history["val_acc"]))
print(f"best validation accuracy: {max(history['val_acc']):.1%} at epoch {best_epoch + 1}")
print(f"validation accuracy at the end (epoch 150): {history['val_acc'][-1]:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
axes[0].plot(history["loss"], alpha=0.4)
axes[0].set_yscale("log"); axes[0].set_xlabel("step"); axes[0].set_title("training loss")
axes[1].plot(history["train_acc"], label="train (noisy labels)")
axes[1].plot(history["val_acc"], label="validation")
axes[1].axvline(best_epoch, color="k", linestyle="--", label="best checkpoint")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("accuracy")
plt.show()

**What we see:** training accuracy climbs toward 100% (so the network memorizes the wrong labels too), while validation accuracy **peaks early, then falls**. If we'd kept only the final model, we'd have lost several points. Keeping the best checkpoint is free.

## 3. Choosing hyperparameters: grid search vs. random search

* **Grid search:** pick a few values for each hyperparameter and try every combination.
* **Random search:** sample each trial's values at random from a range.

**Random search is usually better** (Bergstra & Bengio, 2012), because some hyperparameters matter much more than others. A 3×3 grid tries only **3 different values** of the important one; 9 random trials try **9**. A toy example where only hyperparameter `a` matters:

In [ ]:
def performance(a, b):
    return np.exp(-((a - 0.37) / 0.08)**2) + 0.02 * b       # a sharp peak in `a`; `b` barely matters

grid_values = [1/6, 1/2, 5/6]
grid_best = max(performance(a, b) for a in grid_values for b in grid_values)

random_bests = []
for repeat in range(1000):
    trials = rng.random((9, 2))                              # 9 random (a, b) pairs
    random_bests.append(max(performance(a, b) for a, b in trials))

print(f"3x3 grid search: best {grid_best:.2f}")
print(f"9 random trials: best {np.mean(random_bests):.2f} on average (over 1000 repeats)")

The grid's three values of `a` all happen to miss the peak; random search spreads its tries across `a`.

## 4. Johnson's 7-step recipe for choosing hyperparameters

### Step 1: Check the initial loss
Turn off regularization and check the loss at the start. For softmax with $C$ classes it should be about $\log C$. If not, there's a bug.

In [ ]:
net = Network([64, 128, 10], reg=0.0)
print(f"initial loss: {net.loss(X_train, y_train):.2f}   (expected about log(10) = {np.log(10):.2f})")

### Step 2: Overfit a small sample
Try to get **100% training accuracy on a tiny subset** (a few minibatches), with no regularization. If you can't, something is broken:
* loss not going down → learning rate too low, or bad initialization;
* loss explodes to `inf` or `nan` → learning rate too high.

In [ ]:
tiny_X, tiny_y = X_train[:50], y_train[:50]
for lr in [1e-4, 0.1, 10.0]:
    net = Network([64, 128, 10], reg=0.0)
    history = train(net, tiny_X, tiny_y, epochs=100, batch_size=10, lr=lr)
    print(f"learning rate {lr:<6g}: loss {history['loss'][0]:.2f} -> {history['loss'][-1]:.3g},  "
          f"training accuracy on 50 examples: {history['train_acc'][-1]:.0%}")

Each line shows one diagnosis: too low (the loss barely moves), about right (100%), too high (the loss gets *worse* and the network gets stuck; with other settings it can blow up to `nan`).

### Step 3: Find a learning rate that makes the loss drop
Use all the training data and a small regularization. Try a few learning rates (like 1e-1, 1e-2, 1e-3, 1e-4) for about 100 steps each, and pick one where the loss drops quickly:

In [ ]:
for lr in [1e-4, 1e-3, 1e-2, 1e-1]:
    net = Network([64, 128, 10])
    history = train(net, X_train, y_train, epochs=4, lr=lr)        # 4 epochs x 31 steps, about 100 steps
    print(f"learning rate {lr:<6g}: loss after ~100 steps = {np.mean(history['loss'][-10:]):.3f}")

### Steps 4 and 5: Coarse search, then fine search
**Step 4:** try random learning rates and regularization strengths **around** what worked, training each for just a few epochs. **Step 5:** take the best region, search more narrowly, and train longer. Sample on a **log scale**, since these hyperparameters matter in multiples (0.001 vs. 0.01), not in steps (0.001 vs. 0.002).

In [ ]:
def random_search(n_trials, lr_range, reg_range, epochs, seed):
    rng_search = np.random.default_rng(seed)
    results = []
    for trial in range(n_trials):
        lr = 10 ** rng_search.uniform(*lr_range)            # log-uniform sampling
        reg = 10 ** rng_search.uniform(*reg_range)
        net = Network([64, 128, 10], reg=reg)
        history = train(net, X_train, y_train, epochs=epochs, lr=lr)
        results.append((history["val_acc"][-1], lr, reg))
    return sorted(results, reverse=True)

print("coarse search (3 epochs each):")
coarse = random_search(15, lr_range=(-4, 0), reg_range=(-6, -1), epochs=3, seed=1)
for acc, lr, reg in coarse[:3]:
    print(f"   validation {acc:.1%}   lr {lr:.1e}   reg {reg:.1e}")

best_lr = coarse[0][1]
print("fine search near the best learning rate (15 epochs each):")
fine = random_search(6, lr_range=(np.log10(best_lr) - 0.5, np.log10(best_lr) + 0.5), reg_range=(-6, -2), epochs=15, seed=2)
for acc, lr, reg in fine[:3]:
    print(f"   validation {acc:.1%}   lr {lr:.1e}   reg {reg:.1e}")

### Step 6: Look at the learning curves
What the curves tell you (Johnson's catalog):

| What you see | What it means |
|---|---|
| Loss flat at the start, then suddenly drops | Bad initialization |
| Loss explodes, or gets stuck near the start value after wild steps | Learning rate too high |
| Loss drops, then plateaus too early | Try learning-rate decay |
| **Big gap** between training and validation accuracy | **Overfitting:** more regularization or more data |
| **No gap**, but both low | **Underfitting:** train longer, or use a bigger model |
| Validation accuracy still rising | Train longer! |

Two of these, made on purpose:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))

bad_init = Network([64, 128, 128, 10], init_scale=0.01)          # weights 100x too small
history = train(bad_init, X_train, y_train, epochs=40, lr=0.05)
axes[0].plot(history["loss"], alpha=0.5)
axes[0].set_title("bad initialization: flat, then dropping"); axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")

overfit = Network([64, 128, 128, 10], reg=0.0)
history = train(overfit, X_train[:200], y_train[:200], epochs=40, lr=0.05)
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="validation")
axes[1].set_title("overfitting: a big gap"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.show()
print(f"overfitting example: final train {history['train_acc'][-1]:.0%}, validation {history['val_acc'][-1]:.0%}")

### Step 7: Go back to Step 5
Refine, train longer, add a learning-rate schedule, and repeat.

**One more thing to watch:** the ratio of the **size of each weight update** to the **size of the weights**. Somewhere around **0.001** is a reasonable rule of thumb. Much smaller means learning is crawling; much larger can mean trouble. It's a rough guide, not a target.

## 5. After training: model ensembles

1. Train several models independently (e.g., different random seeds).
2. At test time, **average their predicted probabilities**.

On ImageNet this typically gives about **2%** extra accuracy.

In [ ]:
def probabilities(net, X):
    s = net.scores(X)
    p = np.exp(s - s.max(1, keepdims=True))
    return p / p.sum(1, keepdims=True)

members = []
for seed in range(5):
    net = Network([64, 128, 10], seed=seed)
    train(net, X_train[:300], y_train[:300], epochs=60, lr=0.03, seed=seed)
    members.append(net)

individual = [net.accuracy(X_test, y_test) for net in members]
average_probs = sum(probabilities(net, X_test) for net in members)
print("individual models:", [f"{a:.1%}" for a in individual])
print(f"ensemble of 5:      {np.mean(np.argmax(average_probs, axis=1) == y_test):.1%}")

Here the gain is small: five small networks trained on the same easy digits mostly make the **same** mistakes, and ensembles only help when the members make **different** ones. With big networks on hard data, different seeds end up in genuinely different solutions, and the gain is larger.

**Cheaper tricks:**
* **Snapshot ensembles:** save copies of one model at several points during training (with a cyclic learning rate) and ensemble those.
* **Weight averaging (EMA, or "Polyak averaging"):** keep a running average of the weights during training and use it at test time. It smooths out the noise of the last few steps.

## 6. Transfer learning

**"You need a lot of data to train a CNN." Not necessarily!**
1. **Train on a big dataset** (like ImageNet).
2. **Small new dataset → feature extraction:** keep the trained network, **freeze** it, and train only a **new last layer**.
3. **Bigger new dataset → fine-tuning:** start from the trained network and keep training **all** the layers, with a **smaller learning rate** (about 1/10).

| | Similar to the big dataset | Very different |
|---|---|---|
| **Very little data** | Linear classifier on the top layer | You're in trouble… try features from earlier layers |
| **Quite a lot of data** | Fine-tune a few layers | Fine-tune more layers |

**Our version:** pretrain on digits **0–4** (plenty of data), then learn digits **5–9** from only **50 labeled examples** (10 of each).

In [ ]:
is_source = y_train < 5
X_source, y_source = X_train[is_source], y_train[is_source]          # the "big" dataset: digits 0-4

rng = np.random.default_rng(0)
target_examples = []
for digit in range(5, 10):
    candidates = np.where(y_train == digit)[0]
    target_examples.extend(rng.choice(candidates, 10, replace=False))
X_target, y_target = X_train[target_examples], y_train[target_examples] - 5   # relabel 5..9 as 0..4
is_target_test = y_test >= 5
X_target_test, y_target_test = X_test[is_target_test], y_test[is_target_test] - 5

pretrained = Network([64, 128, 128, 5])
train(pretrained, X_source, y_source, epochs=30, lr=0.05)
print(f"pretrained on digits 0-4 ({len(X_source)} images); new task: {len(X_target)} labeled images of digits 5-9")

Now three ways to learn the new task. To "reuse" the pretrained network, we copy its weights and replace its last layer with a fresh one:

In [ ]:
import copy

def with_new_last_layer(net, seed):
    new_net = copy.deepcopy(net)
    rng_head = np.random.default_rng(seed)
    n_hidden = new_net.W[-1].shape[0]
    new_net.W[-1] = rng_head.normal(0, np.sqrt(2 / n_hidden), size=(n_hidden, 5))
    new_net.b[-1] = np.zeros(5)
    return new_net

results = {"from scratch": [], "frozen features + new last layer": [], "fine-tune everything (lr / 10)": []}
for seed in range(5):
    scratch = Network([64, 128, 128, 5], seed=seed)
    train(scratch, X_target, y_target, epochs=100, batch_size=10, lr=0.05, seed=seed)
    results["from scratch"].append(scratch.accuracy(X_target_test, y_target_test))

    frozen = with_new_last_layer(pretrained, seed)
    frozen.frozen = [0, 1]                                  # don't update the pretrained layers
    train(frozen, X_target, y_target, epochs=100, batch_size=10, lr=0.05, seed=seed)
    results["frozen features + new last layer"].append(frozen.accuracy(X_target_test, y_target_test))

    tuned = with_new_last_layer(pretrained, seed)
    train(tuned, X_target, y_target, epochs=100, batch_size=10, lr=0.005, seed=seed)
    results["fine-tune everything (lr / 10)"].append(tuned.accuracy(X_target_test, y_target_test))

for name, accs in results.items():
    print(f"{name:34s}: {np.mean(accs):.1%} on test digits 5-9 (average of 5 runs)")

**What we see, honestly:** on this toy problem, **training from scratch wins**. Transfer learning pays off when the pretraining data is **much bigger and richer** than what you can learn from your own labels. Our "big" dataset is only about 500 tiny images, and these digits are easy enough that 50 labels already get a fresh network into the 90s. With ImageNet's 1.3 million photos, the result flips: features learned on ImageNet beat training from scratch on almost any small image dataset.

(Notice also that **fine-tuning beats freezing**: the top layer learned features specialized for telling 0–4 apart, and letting it adapt helps. Lower layers learn more general features that transfer better.)

**Transfer learning is everywhere:** object detection, image captioning, and nearly every modern vision system start from a pretrained network. One caveat (He et al., 2018): with enough data and training time, training from scratch can catch up; pretraining mainly **saves time** and helps most when **data is scarce**.

## 7. Large-batch training

With many GPUs (Lecture 9), the total batch size grows. **Goyal et al. (2017)** trained ResNet-50 on ImageNet **in one hour** with batch size 8,192 on 256 GPUs, using two rules:
* **Linear scaling rule:** multiply the batch size by $k$ → **multiply the learning rate by $k$**.
* **Warmup:** a huge learning rate at the start can blow up, so **increase it gradually** over the first few epochs.

Test: batch 16 vs. batch 256, all for **20 epochs**. With 1,000 images, batch 256 takes only 3 steps per epoch instead of 62.

In [ ]:
base_lr = 0.02
warmup = lambda step: 16 * base_lr * min(1.0, (step + 1) / 10)       # ramp up over the first 10 steps
settings = [("batch 16,  lr 0.02 (baseline)", 16, base_lr, None),
            ("batch 256, lr 0.02 (not scaled)", 256, base_lr, None),
            ("batch 256, lr 0.32 (scaled 16x)", 256, 16 * base_lr, None),
            ("batch 256, lr 0.32 with warmup", 256, None, warmup)]
for name, batch_size, lr, schedule in settings:
    accs = []
    for seed in range(3):
        net = Network([64, 128, 10], seed=seed)
        train(net, X_train, y_train, epochs=20, batch_size=batch_size, lr=lr, lr_schedule=schedule, seed=seed)
        accs.append(net.accuracy(X_val, y_val))
    print(f"{name:34s}: validation {np.mean(accs):.1%}")

**What we see:** with the same learning rate, the big batch falls behind (far fewer steps); scaling the learning rate recovers most of the gap. Warmup is a safety measure that matters most when the scaled learning rate is near the edge of what's stable.

## 8. In PyTorch (optional, run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch torchvision
import torch, torch.nn as nn, torchvision

model = torchvision.models.resnet18(weights="IMAGENET1K_V1")        # pretrained on ImageNet
for p in model.parameters():                                        # feature extraction: freeze everything...
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 10)                     # ...and add a new last layer for 10 classes
optimizer = torch.optim.SGD(model.fc.parameters(), lr=1e-2, momentum=0.9)

# Fine-tuning instead: unfreeze, and use a smaller learning rate
for p in model.parameters():
    p.requires_grad = True
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# Early stopping / best checkpoint, inside the training loop:
#   if val_acc > best_val_acc:
#       best_val_acc = val_acc
#       torch.save(model.state_dict(), "best_model.pt")

## Summary
* **Learning rate:** find a good constant one first, then add decay.
* **Early stopping:** train long, keep the best-validation checkpoint.
* **Random search** beats grid search; sample learning rate and regularization on a **log scale**; go **coarse to fine**.
* **The recipe:** check the initial loss → overfit a tiny sample → find a learning rate → coarse search → fine search → read the curves → repeat.
* **Ensembles** average several models; **EMA** averages weights over time.
* **Transfer learning:** reuse a network trained on a big dataset (freeze for little data, fine-tune for more). It shines when your own data is scarce and the pretraining data is large.
* **Large batches:** scale the learning rate with the batch size, and warm up.

### Check your understanding
1. Why does random search usually beat grid search?
2. What does it mean if you can't overfit 50 examples?
3. Training accuracy is 100% and validation accuracy is 80%. What do you do?
4. You increase the batch size from 256 to 2,048. What should you do to the learning rate?

<details><summary>Answers</summary>

1. Usually only a few hyperparameters matter; random search tries many different values of each, while a grid repeats the same few.
2. Something is broken: a bug, a bad initialization, or a learning rate far too low or high.
3. That's overfitting: add regularization (weight decay, dropout, augmentation) or get more data.
4. Multiply it by 8 (the linear scaling rule), and add warmup.
</details>

**Next:** Lecture 12, recurrent networks.